In [ ]:
!uv pip install faiss-gpu-cu12

Using Python 3.12.13 environment at: /usr
Resolved 6 packages in 364ms
Prepared 4 packages in 11.94s
Installed 4 packages in 8ms
 + faiss-gpu-cu12==1.14.1.post1
 + nvidia-cublas-cu12==12.9.2.10
 + nvidia-cuda-nvrtc-cu12==12.9.86
 + nvidia-cuda-runtime-cu12==12.9.79


# **Giai đoạn 1: Embedding**

In [ ]:
import os
import torch
import faiss
import numpy as np
import threading
import queue
import gc  # Thêm thư viện dọn rác thủ công
from PIL import Image
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoProcessor, SiglipModel

# ==========================================
# CẤU HÌNH HỆ THỐNG
# ==========================================
class EmbeddingPipeline:
    def __init__(self, model_name="google/siglip-so400m-patch14-224"):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"[HỆ THỐNG] Khởi tạo {model_name} trên {self.device.upper()}...")

        self.model = SiglipModel.from_pretrained(
            model_name,
            torch_dtype=torch.float16
        ).to(self.device)

        self.processor = AutoProcessor.from_pretrained(model_name)

        # Cấu hình FAISS: IndexFlatL2 + ID Map
        self.dimension = 1152
        self.quantizer = faiss.IndexFlatL2(self.dimension)
        self.index = faiss.IndexIDMap2(self.quantizer)

    def run(self, batch_size=1024):
        ds = load_dataset("tung213/vietnamese-vqa_merged", split="train", streaming=True)
        data_queue = queue.Queue(maxsize=2)

        def prefetch_worker():
            local_images, local_ids = [], []
            for i, example in enumerate(ds):
                try:
                    img = example['image'].convert("RGB")
                    local_images.append(img)
                    local_ids.append(i)
                except: continue

                if len(local_images) == batch_size:
                    # FIX 1: Chạy Processor ngay trong luồng này để nén dữ liệu
                    inputs = self.processor(images=local_images, return_tensors="pt")
                    pixel_tensor = inputs['pixel_values'].to(torch.float16) # Chuyển thành float16

                    # Chỉ đưa Tensor nhỏ gọn vào Queue
                    data_queue.put((pixel_tensor, local_ids))

                    # FIX 2: Tiêu diệt ảnh gốc PIL ngay lập tức để giải phóng RAM
                    for image in local_images:
                        image.close()

                    del local_images, local_ids, inputs
                    local_images, local_ids = [], []

            # Xử lý phần lẻ cuối cùng
            if local_images:
                inputs = self.processor(images=local_images, return_tensors="pt")
                pixel_tensor = inputs['pixel_values'].to(torch.float16)
                data_queue.put((pixel_tensor, local_ids))
                for image in local_images:
                    image.close()
                del local_images, local_ids, inputs

            data_queue.put(None)

        thread = threading.Thread(target=prefetch_worker, daemon=True)
        thread.start()

        print(f"[HỆ THỐNG] Bắt đầu Embedding...")
        with tqdm(total=216450, desc="Embedding Progress") as pbar:
            while True:
                batch_data = data_queue.get()
                if batch_data is None: break

                pixel_tensor, ids = batch_data
                self._process_and_add(pixel_tensor, ids)
                pbar.update(len(ids))

                # FIX 3: Ép Python dọn dẹp bộ nhớ RAM sau mỗi lô
                del pixel_tensor, ids, batch_data
                gc.collect()

    def _process_and_add(self, pixel_tensor, ids):
        # Bắn thẳng Tensor đã chế biến vào GPU
        pixel_tensor = pixel_tensor.to(self.device, non_blocking=True)

        with torch.no_grad():
            outputs = self.model.get_image_features(pixel_values=pixel_tensor)

            if hasattr(outputs, "pooler_output"):
                image_features = outputs.pooler_output
            else:
                image_features = outputs

            image_features = image_features / image_features.norm(dim=-1, keepdim=True)

        # Chuyển về CPU để nạp vào FAISS
        vectors = image_features.cpu().numpy().astype('float32')
        ids_array = np.array(ids).astype('int64')
        self.index.add_with_ids(vectors, ids_array)

        # Dọn dẹp vRAM
        del pixel_tensor, outputs, image_features, vectors, ids_array

    def save(self, filename="vectors.index"):
        faiss.write_index(self.index, filename)
        print(f"[HỆ THỐNG] Đã lưu Index: {filename} ({self.index.ntotal} vectors)")

# ==========================================
# THỰC THI
# ==========================================
pipeline = EmbeddingPipeline()
pipeline.run(batch_size=512)
pipeline.save()

Using Python 3.12.13 environment at: /usr
Checked 1 package in 398ms
[HỆ THỐNG] Khởi tạo google/siglip-so400m-patch14-224 trên CUDA...


Loading weights:   0%|          | 0/888 [00:00<?, ?it/s]

The image processor of type `SiglipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Resolving data files:   0%|          | 0/150 [00:00<?, ?it/s]

[HỆ THỐNG] Bắt đầu Embedding...


Embedding Progress:   0%|          | 0/216450 [00:00<?, ?it/s]

In [ ]:
import faiss
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

class HierarchicalClusterer:
    def __init__(self, index_path="vectors.index"):
        """
        Giai đoạn 2: Khởi tạo hệ thống gom cụm tự động.
        Đọc trực tiếp dữ liệu từ đĩa cứng lên RAM.
        """
        print("[HỆ THỐNG] Đang nạp Vector Database từ đĩa cứng...")
        self.index = faiss.read_index(index_path)
        self.total_vectors = self.index.ntotal
        self.dimension = self.index.d
        print(f"[HỆ THỐNG] Đã tải {self.total_vectors} vectors (Chiều: {self.dimension}).")

        # Bóc tách vector và ID thô ra khỏi lớp vỏ bọc IndexIDMap2
        # Bước này bắt buộc để có thể tái sử dụng vector cho các cụm K-Means con
        flat_index = faiss.downcast_index(self.index.index)
        self.vectors = flat_index.reconstruct_n(0, self.total_vectors)
        self.ids = faiss.vector_to_array(self.index.id_map)

    def run_clustering(self, l1_k=20, l2_k=10):
        """
        Thực thi thuật toán gom cụm phân cấp hai tầng.
        - l1_k: Số lượng đại lục lớn.
        - l2_k: Số lượng quốc gia nhỏ bên trong mỗi đại lục.
        Tổng số cụm tối đa = l1_k * l2_k
        """
        print(f"\n[BƯỚC 1] Bắt đầu phân chia {l1_k} cụm lớn (Macro Clusters)...")
        # Khởi tạo FAISS K-Means. gpu=True giúp tăng tốc độ lên hàng chục lần
        # niter=50 đảm bảo các tâm cụm có đủ thời gian để hội tụ về vị trí chuẩn xác nhất
        kmeans_l1 = faiss.Kmeans(d=self.dimension, k=l1_k, niter=50, verbose=True, gpu=True)
        kmeans_l1.train(self.vectors)

        # Dò tìm xem mỗi bức ảnh đang nằm ở cụm lớn nào
        _, l1_assignments = kmeans_l1.index.search(self.vectors, 1)
        l1_assignments = l1_assignments.flatten()

        results = []

        print(f"\n[BƯỚC 2] Bắt đầu đào sâu vào từng cụm lớn để chia thành {l2_k} cụm nhỏ (Micro Clusters)...")
        # Sử dụng tqdm để theo dõi tiến trình xử lý từng cụm lớn
        for macro_id in tqdm(range(l1_k), desc="Đào sâu Level 2"):
            # Lọc mảng để chỉ lấy các vector thuộc về cụm lớn hiện tại
            mask = (l1_assignments == macro_id)
            sub_vectors = self.vectors[mask]
            sub_ids = self.ids[mask]

            # Cơ chế chống lỗi: Nếu một cụm bị nhiễu và chứa quá ít ảnh,
            # thuật toán sẽ bỏ qua việc chia nhỏ để tránh lỗi toán học
            if len(sub_vectors) <= l2_k:
                for img_id in sub_ids:
                    results.append({"image_id": img_id, "l1_cluster": macro_id, "l2_cluster": 0})
                continue

            # Đào sâu bằng K-Means lần 2 lên chính các vector con
            kmeans_l2 = faiss.Kmeans(d=self.dimension, k=l2_k, niter=30, verbose=False, gpu=True)
            kmeans_l2.train(sub_vectors)

            _, l2_assignments = kmeans_l2.index.search(sub_vectors, 1)
            l2_assignments = l2_assignments.flatten()

            # Ghi nhận kết quả định tuyến của từng bức ảnh
            for img_id, micro_id in zip(sub_ids, l2_assignments):
                results.append({
                    "image_id": img_id,
                    "l1_cluster": macro_id,
                    "l2_cluster": micro_id
                })

        # Định dạng và xuất dữ liệu ra file phục vụ Giai đoạn 3
        df = pd.DataFrame(results)
        df.to_csv("hierarchical_clusters.csv", index=False)
        print("\n[HỆ THỐNG] Hoàn tất! Đã lưu bản đồ phân cấp vào 'hierarchical_clusters.csv'.")

        return df

# ==========================================
# THỰC THI GIAI ĐOẠN 2
# ==========================================
clusterer = HierarchicalClusterer(index_path="vectors.index")

# Bạn có thể tự do điều chỉnh l1_k và l2_k.
# Ví dụ: l1_k=30, l2_k=15 sẽ tạo ra 450 cụm dữ liệu siêu nhỏ lẻ.
df_results = clusterer.run_clustering(l1_k=30, l2_k=15)

[HỆ THỐNG] Đang nạp Vector Database từ đĩa cứng...
[HỆ THỐNG] Đã tải 216450 vectors (Chiều: 1152).

[BƯỚC 1] Bắt đầu phân chia 30 cụm lớn (Macro Clusters)...

[BƯỚC 2] Bắt đầu đào sâu vào từng cụm lớn để chia thành 15 cụm nhỏ (Micro Clusters)...


Đào sâu Level 2:   0%|          | 0/30 [00:00<?, ?it/s]


[HỆ THỐNG] Hoàn tất! Đã lưu bản đồ phân cấp vào 'hierarchical_clusters.csv'.


Bạn cần tạo một tệp `kaggle.json` với thông tin xác thực API của bạn. Kaggle CLI sẽ tự động tìm kiếm tệp này trong thư mục `~/.kaggle/`. Đoạn mã dưới đây sẽ lấy khóa API của bạn từ Colab Secrets và tạo tệp `kaggle.json`.

In [ ]:
# Cài đặt Kaggle API key từ Colab Secrets
from google.colab import userdata
import json
import os

# Lấy tên biến bí mật của bạn (thay thế nếu bạn đặt tên khác)
KAGGLE_USERNAME = userdata.get('KAGGLE_USERNAME')
KAGGLE_KEY = userdata.get('KAGGLE_KEY')

if not KAGGLE_USERNAME or not KAGGLE_KEY:
    print("Lỗi: Vui lòng thêm KAGGLE_USERNAME và KAGGLE_KEY vào Colab Secrets của bạn.")
else:
    # Tạo thư mục .kaggle nếu nó chưa tồn tại
    !mkdir -p ~/.kaggle

    # Tạo tệp kaggle.json
    kaggle_json_content = {"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}
    with open('/root/.kaggle/kaggle.json', 'w') as f:
        json.dump(kaggle_json_content, f)

    # Đặt quyền cho tệp (quan trọng để bảo mật)
    !chmod 600 /root/.kaggle/kaggle.json
    print("Đã cấu hình Kaggle API key thành công.")


Đã cấu hình Kaggle API key thành công.


Bây giờ, bạn có thể thử chạy lại lệnh Kaggle kernel output:

In [ ]:
!kaggle kernels output bnquys/discovery-domain -p .

Output file downloaded to ./vectors.index
Kernel log downloaded to ./discovery-domain.log 


In [ ]:
import pandas as pd

df = pd.read_csv("hierarchical_clusters.csv")
display(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 216450 entries, 0 to 216449
Data columns (total 3 columns):
 #   Column      Non-Null Count   Dtype
---  ------      --------------   -----
 0   image_id    216450 non-null  int64
 1   l1_cluster  216450 non-null  int64
 2   l2_cluster  216450 non-null  int64
dtypes: int64(3)
memory usage: 5.0 MB


None

In [ ]:
import torch
import faiss
import numpy as np
import pandas as pd
from transformers import AutoProcessor, SiglipModel

class Phase3ReverseLabeling:
    def __init__(self, index_path="vectors.index", cluster_csv="hierarchical_clusters.csv"):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.index_path = index_path
        self.cluster_csv = cluster_csv

        # Cuốn từ điển bối cảnh (Domain Vocabulary)
        self.domain_vocabulary = [
            # Nông nghiệp & Thiên nhiên
            "traditional wet-rice cultivation and paddy fields",
            "livestock farming including cattle, pigs and poultry",
            "industrial plantations for coffee, rubber or fruit trees",
            "scenic mountain ranges and forested areas",
            "coastal beaches and marine environments",
            # Đô thị & Đường phố
            "busy traditional wet markets and street vendors",
            "daily pedestrian life on urban sidewalks",
            "metropolitan city centers with modern architecture",
            "residential neighborhoods and narrow city alleys",
            "commuters and traffic on city roads",
            # Trong nhà & Công sở
            "professional office cubicles and corporate meeting rooms",
            "public institutional spaces like banks or post offices",
            "educational settings including classrooms and libraries",
            "indoor residential house kitchens and living rooms",
            "medical clinics and hospital laboratory settings",
            # Dữ liệu số & Rác
            "scanned text documents, receipts and printed papers",
            "digital infographics, data charts and diagrams",
            "screenshots of digital user interfaces and websites",
            "artistic sketches, comics and illustrations",
            "portrait photography focusing on a person's face"
        ]

    def execute_pipeline(self):
        print("[HỆ THỐNG] Bắt đầu quá trình Dò từ điển ngược (Reverse Vocabulary Search)...")

        # Bước 1: Tính toán vector tâm cụm (Centroid)
        centroids_dict = self._calculate_cluster_centroids()

        # Bước 2: So khớp tâm cụm với từ điển bằng SigLIP
        cluster_labels = self._match_centroids_to_vocabulary(centroids_dict)

        # Bước 3: Áp dụng nhãn cho toàn bộ ảnh và lưu kết quả
        self._assign_labels_to_dataset(cluster_labels)

    def _calculate_cluster_centroids(self):
        print("\n[BƯỚC 1] Đang tính toán tâm cụm (Centroids) từ cơ sở dữ liệu vector...")
        df = pd.read_csv(self.cluster_csv)
        index = faiss.read_index(self.index_path)

        flat_index = faiss.downcast_index(index.index)
        all_vectors = flat_index.reconstruct_n(0, index.ntotal)
        all_ids = faiss.vector_to_array(index.id_map)

        id_to_idx = {img_id: idx for idx, img_id in enumerate(all_ids)}
        centroids_dict = {}

        grouped = df.groupby(['l1_cluster', 'l2_cluster'])
        for name, group in grouped:
            group_ids = group['image_id'].values
            valid_indices = [id_to_idx[img_id] for img_id in group_ids if img_id in id_to_idx]

            if not valid_indices:
                continue

            cluster_vectors = all_vectors[valid_indices]
            # Lấy giá trị trung bình để tạo ra vector đại diện cho cả cụm
            centroid = np.mean(cluster_vectors, axis=0)

            # Chuẩn hóa lại vector tâm cụm (L2 Normalization)
            centroid_norm = centroid / np.linalg.norm(centroid)
            centroids_dict[name] = centroid_norm

        return centroids_dict

    def _match_centroids_to_vocabulary(self, centroids_dict):
        print("\n[BƯỚC 2] Khởi tạo SigLIP để đối chiếu tâm cụm với từ điển bối cảnh...")
        model_name = "google/siglip-so400m-patch14-224"
        siglip_model = SiglipModel.from_pretrained(model_name, torch_dtype=torch.float16).to(self.device)
        processor = AutoProcessor.from_pretrained(model_name)

        # Mã hóa toàn bộ từ điển thành Text Vectors
        inputs = processor(text=self.domain_vocabulary, return_tensors="pt", padding="max_length").to(self.device)

        with torch.no_grad():
            outputs = siglip_model.get_text_features(**inputs)

            # ĐOẠN MÃ ĐÃ ĐƯỢC SỬA: Trích xuất Tensor từ đối tượng trả về
            if hasattr(outputs, "pooler_output"):
                text_features = outputs.pooler_output
            else:
                text_features = outputs

            # Bây giờ text_features chắc chắn là một Tensor, ta có thể dùng .norm()
            text_features = text_features / text_features.norm(dim=-1, keepdim=True)
            text_features_tensor = text_features.to(torch.float16)

        cluster_keys = list(centroids_dict.keys())
        centroid_matrix = np.array([centroids_dict[k] for k in cluster_keys])
        centroid_tensor = torch.tensor(centroid_matrix, dtype=torch.float16, device=self.device)

        print("[HỆ THỐNG] Đang thực hiện nhân ma trận để dò tìm nhãn phù hợp nhất...")
        # Tính khoảng cách giữa tâm cụm và các từ vựng
        similarities = torch.matmul(centroid_tensor, text_features_tensor.T)

        best_match_indices = similarities.argmax(dim=-1).cpu().numpy()
        confidence_scores = similarities.max(dim=-1).values.cpu().numpy()

        # Ghép nhãn tương ứng cho từng cụm
        cluster_labels = {}
        for i, key in enumerate(cluster_keys):
            vocab_idx = best_match_indices[i]
            cluster_labels[key] = {
                "domain_label": self.domain_vocabulary[vocab_idx],
                "confidence_score": float(confidence_scores[i])
            }

        # Giải phóng bộ nhớ đồ họa
        del siglip_model, text_features_tensor, centroid_tensor, similarities
        torch.cuda.empty_cache()

        return cluster_labels

    def _assign_labels_to_dataset(self, cluster_labels):
        print("\n[BƯỚC 3] Đang gán nhãn hàng loạt cho toàn bộ bộ dữ liệu...")
        df = pd.read_csv(self.cluster_csv)

        # Hàm phụ trợ để map nhãn dựa trên ID cụm
        def get_label_info(row, key_type):
            cluster_key = (row['l1_cluster'], row['l2_cluster'])
            if cluster_key in cluster_labels:
                return cluster_labels[cluster_key][key_type]
            return "unknown"

        df['domain_label'] = df.apply(lambda row: get_label_info(row, 'domain_label'), axis=1)
        df['confidence_score'] = df.apply(lambda row: get_label_info(row, 'confidence_score'), axis=1)

        df.to_csv("labeled.csv", index=False)
        print("[HỆ THỐNG] HOÀN TẤT CHIẾN DỊCH! Dữ liệu đã được gán nhãn siêu tốc và lưu vào 'labeled.csv'.")

# THỰC THI
pipeline = Phase3ReverseLabeling()
pipeline.execute_pipeline()

[HỆ THỐNG] Bắt đầu quá trình Dò từ điển ngược (Reverse Vocabulary Search)...

[BƯỚC 1] Đang tính toán tâm cụm (Centroids) từ cơ sở dữ liệu vector...

[BƯỚC 2] Khởi tạo SigLIP để đối chiếu tâm cụm với từ điển bối cảnh...


config.json:   0%|          | 0.00/588 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.51G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/888 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/394 [00:00<?, ?B/s]

The image processor of type `SiglipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


tokenizer_config.json:   0%|          | 0.00/711 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/798k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

[HỆ THỐNG] Đang thực hiện nhân ma trận để dò tìm nhãn phù hợp nhất...

[BƯỚC 3] Đang gán nhãn hàng loạt cho toàn bộ bộ dữ liệu...
[HỆ THỐNG] HOÀN TẤT CHIẾN DỊCH! Dữ liệu đã được gán nhãn siêu tốc và lưu vào 'labeled.csv'.


In [ ]:
import pandas as pd

def analyze_labels(file_path="labeled.csv"):
    # Đọc dữ liệu từ tệp tin read CSV
    print(f"[HỆ THỐNG] Đang tải dữ liệu từ {file_path}...")
    try:
        df = pd.read_csv(file_path)
    except FileNotFoundError:
        print(f"[LỖI] Không tìm thấy tệp {file_path}. Hãy kiểm tra lại đường dẫn!")
        return

    # Kiểm tra xem tệp có dữ liệu hay không
    if df.empty:
        print("[LỖI] Tệp dữ liệu trống rỗng. Hãy kiểm tra lại quá trình gán nhãn!")
        return

    print("[HỆ THỐNG] Đang thống kê và phân tích nhãn...")

    # Gom nhóm dữ liệu theo nhãn và tính toán các chỉ số group by and aggregate
    stats = df.groupby('domain_label').agg(
        so_luong_anh=('image_id', 'count'),
        diem_tu_tin_trung_binh=('confidence_score', 'mean')
    ).reset_index()

    # Sắp xếp danh sách từ nhãn có nhiều ảnh nhất đến ít ảnh nhất sort values
    stats = stats.sort_values(by='so_luong_anh', ascending=False).reset_index(drop=True)

    # In báo cáo ra màn hình
    print("\n=== BÁO CÁO THỐNG KÊ NHÃN DỮ LIỆU ===")
    print(f"Tổng số ảnh đã được phân tích: {len(df)}")
    print(f"Tổng số phân loại không gian độc nhất: {len(stats)}\n")

    # Hiển thị toàn bộ bảng dữ liệu không bị cắt bớt
    display(stats)

    # Lưu kết quả báo cáo ra một tệp mới save to CSV
    report_path = "label_statistics.csv"
    stats.to_csv(report_path, index=False)
    print(f"\n[HỆ THỐNG] Đã xuất báo cáo thành công vào tệp {report_path}")

# Kích hoạt chương trình
analyze_labels()

[HỆ THỐNG] Đang tải dữ liệu từ labeled.csv...
[HỆ THỐNG] Đang thống kê và phân tích nhãn...

=== BÁO CÁO THỐNG KÊ NHÃN DỮ LIỆU ===
Tổng số ảnh đã được phân tích: 216450
Tổng số phân loại không gian độc nhất: 17



,domain_label,so_luong_anh,diem_tu_tin_trung_binh
0,"livestock farming including cattle, pigs and p...",31872,0.030469
1,coastal beaches and marine environments,26004,0.049265
2,daily pedestrian life on urban sidewalks,23589,0.066003
3,educational settings including classrooms and ...,21720,0.031875
4,public institutional spaces like banks or post...,21123,0.051897
5,"scanned text documents, receipts and printed p...",17817,0.067893
6,portrait photography focusing on a person's face,17270,0.023443
7,"artistic sketches, comics and illustrations",15385,0.029023
8,indoor residential house kitchens and living r...,10502,0.047636
9,scenic mountain ranges and forested areas,9310,0.034172



[HỆ THỐNG] Đã xuất báo cáo thành công vào tệp label_statistics.csv


In [ ]:
import pandas as pd

def get_image_indices_by_label(file_path="labeled.csv", target_label="livestock farming including cattle, pigs and poultry", min_confidence=0.0):
    """
    Hàm này giúp bạn bốc ra danh sách ID ảnh theo nhãn yêu cầu.
    """
    print(f"[HỆ THỐNG] Đang nạp dữ liệu từ {file_path}...")

    try:
        # Nạp dữ liệu vào DataFrame
        df = pd.read_csv(file_path)
    except FileNotFoundError:
        print(f"[LỖI] Không tìm thấy file {file_path}. Bạn kiểm tra lại đường dẫn nhé!")
        return []

    # Sàng lọc (filtering) dữ liệu theo nhãn và điểm tự tin
    # Chúng ta dùng str.contains để tìm kiếm linh hoạt hơn nếu nhãn dài
    filtered_df = df[
        (df['domain_label'] == target_label) &
        (df['confidence_score'] >= min_confidence)
    ]

    # Nhặt ra cột ID và chuyển thành danh sách (list)
    image_ids = filtered_df['image_id'].tolist()

    print(f"\n[KẾT QUẢ] Tìm thấy {len(image_ids)} ảnh thuộc nhóm: '{target_label}'")

    return image_ids

# --- CẤU HÌNH VÀ CHẠY ---
# Bạn có thể thay đổi nhãn (label) và điểm tự tin (confidence) ở đây
MY_LABEL = "livestock farming including cattle, pigs and poultry"
MIN_SCORE = 0.13  # Chỉ lấy những ảnh mà AI chắc chắn trên 2%

ids_list = get_image_indices_by_label(
    file_path="labeled.csv",
    target_label=MY_LABEL,
    min_confidence=MIN_SCORE
)

# In thử 10 ID đầu tiên để kiểm tra
if ids_list:
    print(f"10 ID đầu tiên trong danh sách: {ids_list[:10]}")

    # Nếu bạn muốn lưu danh sách này ra file txt để dùng sau:
    # with open("image_ids.txt", "w") as f:
    #     for img_id in ids_list:
    #         f.write(f"{img_id}\n")
    # print("[HỆ THỐNG] Đã lưu danh sách ID vào file 'image_ids.txt'")

[HỆ THỐNG] Đang nạp dữ liệu từ labeled.csv...

[KẾT QUẢ] Tìm thấy 973 ảnh thuộc nhóm: 'livestock farming including cattle, pigs and poultry'
10 ID đầu tiên trong danh sách: [149, 563, 717, 843, 957, 984, 1040, 1213, 1305, 1416]


In [ ]:
import os
import csv
from datasets import load_dataset
from tqdm.auto import tqdm

def download_and_log_realtime(target_ids, output_dir="images", csv_name="images.csv"):
    """
    Tải ảnh và ghi nhật ký vào CSV theo thời gian thực (Incremental logging).
    """
    # 1. Khởi tạo thư mục
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"[HỆ THỐNG] Đã tạo thư mục: {output_dir}")

    target_set = set(target_ids)
    total_to_download = len(target_set)
    TOTAL_DATASET = 216450

    # 2. Kết nối dataset streaming
    ds = load_dataset("tung213/vietnamese-vqa_merged", split="train", streaming=True)

    count_downloaded = 0
    max_target = max(target_set) if target_set else 0

    print("[HỆ THỐNG] Bắt đầu quét và tải ảnh (Ghi CSV thời gian thực)...")

    # 3. Mở file CSV trước khi vào vòng lặp
    with open(csv_name, mode='w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        # Ghi tiêu đề ngay lập tức
        writer.writerow(['image_idx', 'file_name'])

        # Khởi tạo thanh tiến trình song song
        with tqdm(total=TOTAL_DATASET, desc="Quét Dataset", position=0) as pbar_scan:
            with tqdm(total=total_to_download, desc="Ảnh đã tải  ", position=1, leave=True) as pbar_download:

                for i, example in enumerate(ds):
                    pbar_scan.update(1)

                    if i in target_set:
                        file_name = f"{i:06d}.jpg"
                        file_path = os.path.join(output_dir, file_name)

                        try:
                            # Tải và lưu ảnh
                            img = example['image'].convert("RGB")
                            img.save(file_path)

                            # GHI CSV NGAY LẬP TỨC TẠI ĐÂY
                            writer.writerow([i, file_name])

                            # Ép hệ thống đẩy dữ liệu từ bộ đệm xuống đĩa (Optional)
                            # f.flush()

                            count_downloaded += 1
                            pbar_download.update(1)
                        except Exception as e:
                            tqdm.write(f"[CẢNH BÁO] Lỗi tại index {i}: {e}")

                    # Dừng sớm nếu đã đủ
                    if count_downloaded == total_to_download or i >= max_target:
                        pbar_scan.n = pbar_scan.total
                        pbar_scan.refresh()
                        break

    print(f"\n[HOÀN TẤT] Tiến trình kết thúc an toàn. File nhật ký: {csv_name}")

# --- THỰC THI ---
if 'ids_list' in locals() and ids_list:
    download_and_log_realtime(ids_list)

[HỆ THỐNG] Đã tạo thư mục: images


Resolving data files:   0%|          | 0/150 [00:00<?, ?it/s]

[HỆ THỐNG] Bắt đầu quét và tải ảnh (Ghi CSV thời gian thực)...


Quét Dataset:   0%|          | 0/216450 [00:00<?, ?it/s]

Ảnh đã tải  :   0%|          | 0/973 [00:00<?, ?it/s]

KeyboardInterrupt: 